# IFSP — Câmpus Presidente Epitácio
## Bacharelado em Ciência da Computação • Aprendizado de Máquina (PEPAPRM)
### Atividade Prática: Três Bases, Três Problemas
---
# Base 2: Previsão de Preço de Venda de Imóveis (Regressão)
**Pergunta de Negócio:** *Quanto vale um imóvel com estas características em uma cidade média?*


## 1. Identificação do Problema e Justificativa

Aplicando o fluxograma **Tenho um TARGET ($y$)?**:
- A base possui a variável alvo `preco_venda`, que é um valor numérico contínuo em moeda (Reais - R$).
- Trata-se, portanto, de um problema de **Aprendizado Supervisionado de Regressão**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

RANDOM_STATE = 42


## 2. Carregamento e Inspeção Inicial dos Dados


In [ ]:
df = pd.read_csv('base2_regressao_imoveis.csv')
print(f"Dimensões do dataset original: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()


In [ ]:
df.info()


## 3. Perguntas de Análise e Tratamento de Dados

### Pergunta 1: Olhe describe() de cada coluna numérica. Algum valor é fisicamente impossível ou parece erro de digitação? Como você decidiu tratá-lo?
- Ao inspecionar `df.describe()`, nota-se que a coluna `area_m2` possui valor máximo de **2100.0 m²**, enquanto o percentil 75% é de apenas 118.9 m² e a segunda maior área é de 212.9 m².
- Investigando essa linha específica (índice 476), trata-se de um imóvel de 3 quartos, 1 banheiro e valor de R$ 238.205 no bairro Vila Nova. É evidente um **erro de digitação** (um zero extra digitado acidentalmente em 210.0 m²).
- **Tratamento adotado**: Corrigimos o valor pontual para `210.0 m²` para preservar a integridade da amostra.


In [ ]:
df.describe()


In [ ]:
# Identificação e correção do erro de digitação na área
outlier_area = df[df['area_m2'] > 500]
print("Registro com valor discrepante em area_m2:")
print(outlier_area[['area_m2', 'quartos', 'banheiros', 'bairro', 'preco_venda']])

# Correção fundamentada do erro de digitação
df.loc[df['area_m2'] == 2100.0, 'area_m2'] = 210.0
print(f"Novo valor máximo de area_m2 após correção: {df['area_m2'].max():.1f} m²")


### Pergunta 2: Qual é a ordem de grandeza do alvo? Que algoritmos podem ter dificuldade com ela?
- **Ordem de grandeza**: O alvo `preco_venda` varia de R$ 90.000 a mais de R$ 1.400.000 (ordem de magnitude de centenas de milhares, $10^5$ a $10^6$).
- **Dificuldade dos algoritmos**: Algoritmos baseados em descida de gradiente (ex: SGDRegressor, redes neurais) ou baseados em distâncias e regularização (ex: SVR, KNN) podem apresentar instabilidade ou convergência lenta sem uma padronização rigorosa dos atributos de entrada. Algoritmos baseados em árvores (Random Forest, Gradient Boosting) são imunes à escala dos preditores, mas a magnitude do alvo reflete diretamente nas métricas absolutas (MAE e RMSE em reais).


### Pergunta 3: Qual seria o erro de um modelo que sempre prevê a média? Use-o como piso (DummyRegressor).
- O modelo basal prediz a média dos preços de treino para qualquer imóvel. O erro quadrático médio basal estimado é de aproximadamente **R$ 193.850**, estabelecendo o piso mínimo a ser superado pelos modelos reais.


In [ ]:
# Verificação de valores ausentes
print("Valores ausentes por coluna:")
print(df.isnull().sum()[df.isnull().sum() > 0])


## 4. Separação em X, y e Divisão Treino e Teste


In [ ]:
X = df.drop(columns=['preco_venda'])
y = df['preco_venda']

colunas_num = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
colunas_cat = X.select_dtypes(include=['object', 'str']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE
)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")


## 5. Pipeline de Pré-Processamento (Sem Vazamento de Dados)
- Atributos numéricos: imputação pela mediana e padronização com `StandardScaler()`.
- Atributo categórico (`bairro`): imputação pelo mais frequente e codificação com `OneHotEncoder(drop='first')`.


In [ ]:
pipeline_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

pipeline_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', pipeline_num, colunas_num),
    ('cat', pipeline_cat, colunas_cat)
])


## 6. Escolha e Justificativa dos Algoritmos Candidatos (+ Baseline)

1. **DummyRegressor (Baseline)**: Sempre prevê a média do alvo no treino; serve de teto de erro admissível.
2. **Ridge (Candidato 1)**: Regressão Linear com regularização L2, permitindo modelar relações lineares diretas (área, quartos) evitando coeficientes explosivos.
3. **Random Forest Regressor (Candidato 2)**: Ensemble de árvores de decisão não-linear que captura interações entre bairro e tamanho sem premissas lineares rígidas.
4. **HistGradientBoostingRegressor (Candidato 3)**: Modelo baseado em árvores com boosting de gradiente sequencial, com alta capacidade de ajuste e velocidade.


In [ ]:
modelos = {
    'Dummy (Baseline Média)': DummyRegressor(strategy='mean'),
    'Ridge (Linear L2)': Ridge(random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(random_state=RANDOM_STATE),
    'HistGradientBoosting': HistGradientBoostingRegressor(random_state=RANDOM_STATE)
}


## 7. Validação Cruzada K-Fold com RMSE
Configuração: `KFold(n_splits=5, shuffle=True, random_state=42)` com métrica `neg_root_mean_squared_error`.


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

resultados_cv = []
for nome, modelo in modelos.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', modelo)
    ])
    # scikit-learn retorna o RMSE negativo para scoring de maximização
    scores_neg_rmse = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')
    scores_r2 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='r2')
    
    rmse_scores = -scores_neg_rmse
    resultados_cv.append({
        'Modelo': nome,
        'RMSE CV Médio (R$)': rmse_scores.mean(),
        'RMSE CV Std (R$)': rmse_scores.std(),
        'R² CV Médio': scores_r2.mean()
    })

df_cv = pd.DataFrame(resultados_cv)
df_cv


## 8. Otimização de Hiperparâmetros (GridSearchCV)
O `HistGradientBoostingRegressor` obteve o menor RMSE em validação cruzada. Vamos otimizar a taxa de aprendizado e o número de iterações.


In [ ]:
pipe_opt = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(random_state=RANDOM_STATE))
])

param_grid = {
    'regressor__max_iter': [80, 120, 200],
    'regressor__learning_rate': [0.03, 0.05, 0.1],
    'regressor__max_leaf_nodes': [15, 31, 50]
}

grid_search = GridSearchCV(
    estimator=pipe_opt,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f"Melhores hiperparâmetros: {grid_search.best_params_}")
print(f"Melhor RMSE na validação cruzada: R$ {-grid_search.best_score_:.2f}")

melhor_modelo = grid_search.best_estimator_


## 9. Avaliação Final no Conjunto de Teste (Executada Uma Única Vez)


In [ ]:
# Previsão final
y_pred = melhor_modelo.predict(X_test)

# Previsão baseline Dummy para comparação
dummy = Pipeline([('preprocessor', preprocessor), ('regressor', DummyRegressor(strategy='mean'))])
dummy.fit(X_train, y_train)
y_dummy = dummy.predict(X_test)

mae_final = mean_absolute_error(y_test, y_pred)
rmse_final = root_mean_squared_error(y_test, y_pred)
r2_final = r2_score(y_test, y_pred)

mae_dummy = mean_absolute_error(y_test, y_dummy)
rmse_dummy = root_mean_squared_error(y_test, y_dummy)
r2_dummy = r2_score(y_test, y_dummy)

print("=== COMPARAÇÃO FINAL NO TESTE ===")
print(f"{'Métrica':<10} | {'Dummy (Baseline)':<18} | {'Melhor Modelo (Boosting)':<24}")
print("-" * 60)
print(f"{'MAE':<10} | R$ {mae_dummy:15.2f} | R$ {mae_final:20.2f}")
print(f"{'RMSE':<10} | R$ {rmse_dummy:15.2f} | R$ {rmse_final:20.2f}")
print(f"{'R²':<10} | {r2_dummy:18.4f} | {r2_final:24.4f}")


## 10. Diagnóstico de Resíduos e Conclusão

Plotaremos:
1. Preço Real vs Preço Previsto (com a linha de perfeição $y = x$).
2. Gráfico de Resíduos ($y - \hat{y}$) em função do valor previsto.


In [ ]:
residuos = y_test - y_pred

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Real vs Previsto
ax[0].scatter(y_test, y_pred, alpha=0.6, color='royalblue', edgecolors='k', s=40)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
ax[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Previsão Perfeita (y = x)')
ax[0].set_title(f'Preço Real vs Previsto (R² = {r2_final:.3f})')
ax[0].set_xlabel('Preço Real (R$)')
ax[0].set_ylabel('Preço Previsto (R$)')
ax[0].legend()
ax[0].grid(True, linestyle=':', alpha=0.6)

# Gráfico de Resíduos
ax[1].scatter(y_pred, residuos, alpha=0.6, color='forestgreen', edgecolors='k', s=40)
ax[1].axhline(0, color='r', linestyle='--', lw=2)
ax[1].set_title('Gráfico de Resíduos (Real - Previsto)')
ax[1].set_xlabel('Preço Previsto (R$)')
ax[1].set_ylabel('Resíduo (R$)')
ax[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


### Análise dos Resíduos: há algum padrão que o modelo não capturou?
Os resíduos estão distribuídos de maneira aproximadamente simétrica em torno de zero para a ampla maioria dos imóveis (entre R$ 150.000 e R$ 700.000), demonstrando que o modelo capturou muito bem a relação de valor da região. Nota-se apenas uma ligeira heterocedasticidade com maior dispersão em imóveis de altíssimo padrão acima de R$ 900.000, onde acabamentos de luxo específicos e fatores subjetivos não presentes nas colunas explicam a variação residual.
